# Exploring Greenway's home energy bills

A place to look at the data and try your functions. The tests check `explain.py`, not this notebook.

Choose the `.venv` kernel (**Select Kernel**, top right) and run the cells from the top.

In [ ]:
import pandas as pd

homes = pd.read_csv("home_energy.csv")
homes.shape

In [ ]:
homes.head(10)

In [ ]:
homes.describe()

In [ ]:
homes.groupby("insulation")["year_built"].mean().round()

## The model to explain

`bill_model.py` splits the homes and builds the forest. Training it takes a few seconds.

In [ ]:
from sklearn.metrics import mean_absolute_error

from bill_model import build_bill_model, split_homes

X_train, X_test, y_train, y_test = split_homes(homes, 42)
model = build_bill_model(0).fit(X_train, y_train)
mean_absolute_error(y_test, model.predict(X_test))

## Try your functions

After you change `explain.py`, restart the kernel (**Restart** in the notebook's toolbar) and run the cells again, so the notebook imports your newest code.

In [ ]:
from explain import importance_table, plain_summary, top_features

table = importance_table(model, X_test, y_test, 0)
table

In [ ]:
top = top_features(table, 3)
top

In [ ]:
for sentence in plain_summary(top, mean_absolute_error(y_test, model.predict(X_test))):
    print(sentence)

## How the guess changes with one feature

Partial dependence needs the number columns as floats (*How a Guess Changes with One Feature*).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay

X_float = X_train.astype({"size_sqft": float, "year_built": float, "occupants": float})
PartialDependenceDisplay.from_estimator(model, X_float, ["size_sqft", "year_built"], kind="both", subsample=50, random_state=0)
plt.show()